# NB_AP_Conform · Phase 2 · Bronze → LH_AP_Silver
Types, deduplicates and decodes the Payables EBS tables. `silver.supplier` keeps history as SCD Type 2.

In [ ]:
%%configure -f
{"defaultLakehouse": {"name": "LH_AP_Silver"}}

In [ ]:
BRONZE = "abfss://{{WS_AP_BRONZE}}@onelake.dfs.fabric.microsoft.com/{{LH_AP_BRONZE}}/Tables/ebs"
FX_TO_KWD = {"KWD": 1.0, "SAR": 0.0819, "AED": 0.0836, "EUR": 0.335, "USD": 0.307}   # lab rates

In [ ]:
from pyspark.sql import functions as F, Window as W
from delta.tables import DeltaTable
for t in ["ap_suppliers", "ap_supplier_sites_all", "ap_terms", "iby_ext_bank_accounts", "ap_invoices_all", "ap_invoice_lines_all",
          "ap_payment_schedules_all", "ap_checks_all", "ap_invoice_payments_all"]:
    spark.read.format("delta").load(f"{BRONZE}/{t}").createOrReplaceTempView(f"b_{t}")
spark.sql("SELECT * FROM VALUES " + ", ".join(f"('{k}', CAST({v} AS DOUBLE))" for k, v in FX_TO_KWD.items()) + " AS t(currency_code, rate_to_kwd)").createOrReplaceTempView("fx")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
def save(df, name):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"silver.{name}")
    print(f"silver.{name}: {spark.table('silver.' + name).count()} rows")

In [ ]:
# silver.supplier: SCD Type 2 on name, VAT number, category, country and payment terms
src = spark.sql("""
WITH site AS (SELECT VENDOR_ID, FIRST(COUNTRY) country, FIRST(TERMS_ID) terms_id FROM b_ap_supplier_sites_all GROUP BY VENDOR_ID)
SELECT CAST(s.VENDOR_ID AS INT) supplier_id, s.VENDOR_NAME supplier_name, s.VAT_REGISTRATION_NUM vat_registration_num,
       site.country, s.ATTRIBUTE1 category, t.NAME payment_terms
FROM b_ap_suppliers s LEFT JOIN site ON site.VENDOR_ID = s.VENDOR_ID LEFT JOIN b_ap_terms t ON t.TERM_ID = site.terms_id
WHERE s.ENABLED_FLAG = 'Y'""").withColumn("_hash", F.sha2(F.concat_ws("|", "supplier_name", "vat_registration_num", "country", "category", "payment_terms"), 256))
today = F.current_date()
if not spark.catalog.tableExists("silver.supplier"):
    (src.withColumn("valid_from", F.lit("2020-01-01").cast("date")).withColumn("valid_to", F.lit("9999-12-31").cast("date"))
        .withColumn("is_current", F.lit(True)).write.saveAsTable("silver.supplier"))
else:
    tgt = DeltaTable.forName(spark, "silver.supplier")
    cur = spark.table("silver.supplier").filter("is_current")
    changed = src.alias("s").join(cur.alias("c"), "supplier_id", "left").filter("c._hash IS NULL OR c._hash <> s._hash").select("s.*")
    (tgt.alias("t").merge(changed.alias("s"), "t.supplier_id = s.supplier_id AND t.is_current")
        .whenMatchedUpdate(set={"is_current": "false", "valid_to": "date_sub(current_date(), 1)"}).execute())
    (changed.withColumn("valid_from", today).withColumn("valid_to", F.lit("9999-12-31").cast("date")).withColumn("is_current", F.lit(True))
        .write.mode("append").saveAsTable("silver.supplier"))
print("silver.supplier:", spark.table("silver.supplier").count(), "rows,", spark.table("silver.supplier").filter("is_current").count(), "current")

In [ ]:
save(spark.sql("""
SELECT CAST(ss.VENDOR_SITE_ID AS INT) supplier_site_id, CAST(ss.VENDOR_ID AS INT) supplier_id, CAST(ss.ORG_ID AS INT) org_id,
       ss.VENDOR_SITE_CODE site_code, ss.COUNTRY country, ss.CITY city, t.NAME payment_terms
FROM b_ap_supplier_sites_all ss LEFT JOIN b_ap_terms t ON t.TERM_ID = ss.TERMS_ID"""), "supplier_site")

save(spark.sql("""
SELECT CAST(i.INVOICE_ID AS INT) invoice_id, i.INVOICE_NUM invoice_number, CAST(i.VENDOR_ID AS INT) supplier_id,
       CAST(i.VENDOR_SITE_ID AS INT) supplier_site_id, CAST(i.ORG_ID AS INT) org_id, CAST(i.INVOICE_DATE AS DATE) invoice_date,
       CAST(i.INVOICE_AMOUNT AS DECIMAL(18,3)) invoice_amount, i.INVOICE_CURRENCY_CODE currency_code,
       CAST(i.INVOICE_AMOUNT * fx.rate_to_kwd AS DECIMAL(18,3)) invoice_amount_kwd,
       CASE i.PAYMENT_STATUS_FLAG WHEN 'Y' THEN 'Paid' WHEN 'P' THEN 'Partially paid' ELSE 'Unpaid' END payment_status,
       INITCAP(i.APPROVAL_STATUS) approval_status, CAST(ps.DUE_DATE AS DATE) due_date,
       CAST(ps.AMOUNT_REMAINING AS DECIMAL(18,3)) amount_remaining
FROM b_ap_invoices_all i JOIN fx ON fx.currency_code = i.INVOICE_CURRENCY_CODE
LEFT JOIN b_ap_payment_schedules_all ps ON ps.INVOICE_ID = i.INVOICE_ID"""), "invoice")

save(spark.sql("""
SELECT CAST(INVOICE_ID AS INT) invoice_id, CAST(LINE_NUMBER AS INT) line_number, CAST(ORG_ID AS INT) org_id,
       CAST(AMOUNT AS DECIMAL(18,3)) amount, CAST(PO_HEADER_ID AS INT) po_header_id, CAST(PO_LINE_ID AS INT) po_line_id
FROM b_ap_invoice_lines_all"""), "invoice_line")

save(spark.sql("""
SELECT CAST(p.INVOICE_PAYMENT_ID AS INT) invoice_payment_id, CAST(c.CHECK_ID AS INT) check_id, CAST(c.CHECK_NUMBER AS BIGINT) check_number,
       CAST(p.INVOICE_ID AS INT) invoice_id, CAST(c.VENDOR_ID AS INT) supplier_id, CAST(p.ORG_ID AS INT) org_id,
       CAST(p.ACCOUNTING_DATE AS DATE) payment_date, CAST(p.AMOUNT AS DECIMAL(18,3)) amount, c.CURRENCY_CODE currency_code,
       CAST(p.AMOUNT * fx.rate_to_kwd AS DECIMAL(18,3)) amount_kwd, CAST(c.EXTERNAL_BANK_ACCOUNT_ID AS INT) bank_account_id
FROM b_ap_invoice_payments_all p JOIN b_ap_checks_all c ON c.CHECK_ID = p.CHECK_ID JOIN fx ON fx.currency_code = c.CURRENCY_CODE"""), "payment")

save(spark.sql("""
SELECT CAST(EXT_BANK_ACCOUNT_ID AS INT) bank_account_id, CAST(VENDOR_ID AS INT) supplier_id, BANK_NAME bank_name,
       BANK_ACCOUNT_NUM bank_account_num, IBAN iban, CURRENCY_CODE currency_code
FROM b_iby_ext_bank_accounts"""), "bank_account")

In [ ]:
for t in ["supplier_site", "invoice", "invoice_line", "payment"]:
    assert spark.sql(f"SELECT COUNT(*) n FROM silver.{t} WHERE org_id IS NULL").first().n == 0, t
print("Data quality checks passed")